# r3con — quickstart

Ask a question whose evidence is spread across several documents.

The five memos in `memos/` are set up so that **no single one contains the answer**: the
incident counts are in three site reports that name the contractor only by code, the
code→name mapping is in a fourth document that has no counts, and the fifth is about a
cafeteria and is irrelevant.

```
pip install r3context
```

In [1]:
import os

from r3con import r3con

# The key comes from the provider's own variable, or a .env beside this notebook.
assert os.environ.get("OPENAI_API_KEY") or os.path.exists(".env"), "Set OPENAI_API_KEY first."

MODEL = "openai/gpt-5.6-luna"    # any litellm model string

QUESTION = (
    "Which contractor was responsible for the most equipment incidents across our "
    "sites in Q3, and how many? Give the contractor's name, not its code."
)

# The correct answer, so you can check the run:
#
#   Halloran Services Ltd, with 11 incidents.
#
#   CT-118 -> Northgate 5 + Riverside 6 = 11      <- the winner
#   CT-204 -> Eastfield 9
#   and CT-118 is Halloran Services Ltd, per the registry.
#
# Note that no single document contains "Halloran" and a count: the sites report
# counts against a CODE, and only the registry maps that code to a name.

## Documents as a list of strings

In [2]:
DOCS = [
    """MEMO — Northgate site, Q3 operations review

    Equipment incidents logged this quarter: 5.
    Three were conveyor stoppages, two were HVAC trips.
    Planned maintenance at this site is carried out under contractor code CT-118.
    """,
    """MEMO — Eastfield site, Q3 operations review

    Equipment incidents logged this quarter: 9.
    Mostly compressor faults, clustered in August.
    Planned maintenance at this site is carried out under contractor code CT-204.
    """,
    """MEMO — Riverside site, Q3 operations review

    Equipment incidents logged this quarter: 6.
    Four were electrical, two were water ingress after the September storm.
    Planned maintenance at this site is carried out under contractor code CT-118.
    """,
    """APPROVED CONTRACTOR REGISTRY (extract)

    CT-118 .... Halloran Services Ltd — mechanical and electrical maintenance
    CT-204 .... Merrow Facilities Group — HVAC and compressed air
    """,
    """MEMO — Cafeteria refurbishment, phase 2

    The serving counter will be replaced during the October shutdown. Seating
    capacity rises from 60 to 82. No impact on production areas.
    """,
]

result = r3con.run(question=QUESTION, documents=DOCS, model=MODEL)
print(result)

Halloran Services Ltd (CT-118) had the most associated Q3 equipment incidents, with 11 incidents across Northgate and Riverside. The documents identify Halloran as performing planned maintenance, but do not explicitly assign responsibility for causing the incidents.


Halloran Services, with 11 — 5 at Northgate plus 6 at Riverside. That number and
that name never appear in the same document.

## Documents from a folder

`read_documents` turns a folder, a glob, or a list of files into that same list of
strings.

In [3]:
docs = r3con.read_documents("memos")
print(f"{len(docs)} documents read from memos/\n")

5 documents read from memos/



In [4]:
result = r3con.run(question=QUESTION, documents=docs, model=MODEL)
print(result)

Halloran Services Ltd was responsible for the most equipment incidents in Q3, with 11 incidents across Northgate and Riverside.


## What came back

`run()` returns the answer plus the views it was derived from.

In [5]:
print("ANSWER      :", result.answer, "\n")

print("RELEVANCE   — what each document was found to contribute:")
for i, snippet in enumerate(result.relevance, 1):
    first = next(iter(snippet.strip().splitlines()), "(nothing relevant to this task)")
    print(f"  [{i}] {first[:100]}")

print("\nSTRUCT_DATA — the records the answer was computed from:")
for field, rows in result.struct_data.items():
    print(f"  {field}: {len(rows)} records")

print("\nSCHEMA_CODE — designed for this question:")
print("  " + "\n  ".join(result.schema_code.splitlines()[:8]))

print("\nRUN_DIR     :", os.path.relpath(result.run_dir))

ANSWER      : Halloran Services Ltd was responsible for the most equipment incidents in Q3, with 11 incidents across Northgate and Riverside. 

RELEVANCE   — what each document was found to contribute:
  [1] A Q3 Northgate site operations memo reporting 5 equipment incidents—three conveyor stoppages and two
  [2] A Q3 operations memo for the Eastfield site. It reports 9 equipment incidents, mostly compressor fau
  [3] A Riverside site Q3 operations memo reporting 6 equipment incidents: four electrical and two caused 
  [4] An approved contractor registry extract that maps site paperwork codes to contractor names. It ident
  [5] A cafeteria refurbishment memo describing an October serving-counter replacement and expanded seatin

STRUCT_DATA — the records the answer was computed from:
  equipment_incident_reports: 3 records
  contractor_code_mappings: 3 records

SCHEMA_CODE — designed for this question:
  from pydantic import BaseModel, Field
  
  
  class EquipmentIncidentReport(BaseMod

## From the command line

```bash
r3con run "Which contractor had the most incidents in Q3?" memos
```